In [11]:
import mlflow
import pandas as pd

from walmart_forecasting.paths import (
    RAW_DATA_DIR,
)
from walmart_forecasting.tracking import (
    setup_mlflow,
)


setup_mlflow(
    experiment_name="Final_Ensemble_Inference"
)

pipeline = mlflow.pyfunc.load_model(
    "models:/"
    "Walmart-LightGBM-PatchTST-Ensemble"
    "@production"
)

raw_test = pd.read_csv(
    RAW_DATA_DIR / "test.csv"
)

predictions = pipeline.predict(
    raw_test
)

Initialized MLflow to track repo "lkhiz23/walmart-store-sales-forecasting"

Repository lkhiz23/walmart-store-sales-forecasting initialized!

/home/xizusha/Documents/ML/walmart-store-sales-forecasting/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [12]:
print(type(predictions))
print(predictions.shape)

display(predictions.head())
display(predictions.tail())

print(
    predictions["Prediction"].describe()
)

<class 'pandas.core.frame.DataFrame'>
(115064, 1)


,Prediction
0,37010.575349
1,19560.083851
2,19451.466852
3,21625.020276
4,24736.729990


,Prediction
115059,688.243265
115060,682.443382
115061,700.550805
115062,792.369259
115063,663.109613


count    115064.000000
mean      16416.512377
std       23454.254941
min       -2641.088262
25%        2109.816237
50%        7544.510453
75%       20745.595947
max      420001.049758
Name: Prediction, dtype: float64


In [13]:
import numpy as np
import pandas as pd

from walmart_forecasting.paths import (
    RAW_DATA_DIR,
    SUBMISSIONS_DIR,
)


sample_submission = pd.read_csv(
    RAW_DATA_DIR / "sampleSubmission.csv"
)

if len(predictions) != len(sample_submission):
    raise ValueError(
        "Predictions and sample submission "
        "have different row counts."
    )

prediction_values = predictions[
    "Prediction"
].to_numpy(
    dtype=float
)

if not np.isfinite(
    prediction_values
).all():
    raise ValueError(
        "Predictions contain NaN or infinity."
    )

submission = sample_submission.copy()

submission[
    "Weekly_Sales"
] = prediction_values

SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

submission_path = (
    SUBMISSIONS_DIR
    / "lightgbm_patchtst_equal_weight_submission.csv"
)

submission.to_csv(
    submission_path,
    index=False,
)

print(
    "Submission saved to:",
    submission_path
)

print(
    "Submission shape:",
    submission.shape
)

display(submission.head())

Submission saved to: /home/xizusha/Documents/ML/walmart-store-sales-forecasting/reports/submissions/lightgbm_patchtst_equal_weight_submission.csv
Submission shape: (115064, 2)


,Id,Weekly_Sales
0,1_1_2012-11-02,37010.575349
1,1_1_2012-11-09,19560.083851
2,1_1_2012-11-16,19451.466852
3,1_1_2012-11-23,21625.020276
4,1_1_2012-11-30,24736.729990
